# Data

In [27]:
import time

from pathlib import Path
from datasets import Dataset, DatasetDict, Value, ClassLabel, Features
from tqdm.notebook import tqdm

In [2]:
PATH = Path("data/aclImdb")

In [54]:
def create_data(path):
    """
    Create a dataset from text files in a given directory.

    Args:
        path (Path): Path to the directory containing the dataset.

    Returns:
        Dataset: A Hugging Face Dataset object containing the text data and labels.
    """
    splits = ["train", "test"]
    labels = {"pos": 1, "neg": 0}
    data = {
        "train": {
            "text": [],
            "label": [],
        },
        "test": {
            "text": [],
            "label": [],
        }
    }

    for split in tqdm(splits, desc="Splits"):
        for label in tqdm(labels.keys(), desc="Labels", leave=False):
            for file in tqdm(
                iterable=(path / split / label).iterdir(),
                desc="Files",
                total=len(list((path / split / label).iterdir())),
                leave=False):
                if file.is_file():
                    with open(file, "r") as f:
                        for line in f:
                            line = line.strip()
                            if line:
                                data[split]["text"].append(line)
                                data[split]["label"].append(labels[label])
    return data

In [ ]:
data = create_data(PATH)

In [59]:
dataset_dict = DatasetDict({
    'train': Dataset.from_dict(data['train'], features=Features(features)),
    'test': Dataset.from_dict(data['test'],features=Features(features))
})

In [ ]:
dataset_dict.save_to_disk("data/aclImdb_dataset")

In [52]:
ds = DatasetDict.load_from_disk("data/aclImdb_dataset")
print(ds)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 ds = DatasetDict.load_from_disk("data/aclImdb_dataset")                                      │
│   2 print(ds)                                                                                    │
│   3                                                                                              │
│                                                                                                  │
│ /home/ssm-user/.cache/pypoetry/virtualenvs/gda-c7vhl4sO-py3.10/lib/python3.10/site-packages/data │
│ sets/dataset_dict.py:1395 in load_from_disk                                                      │
│                                                                                                  │
│   1392 │   │   │   │   raise FileNotFoundError(                                                  │
│   1393 │   │   │   │   │   f"No such file: '{dataset_dict_json_path}'. Expected to load a `Data  │
│   1394 │   │   │   │   )                                                                         │
│ ❱ 1395 │   │   │   raise FileNotFoundError(                                                      │
│   1396 │   │   │   │   f"No such file: '{dataset_dict_json_path}'. Expected to load a `DatasetD  │
│   1397 │   │   │   )                                                                             │
│   1398                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
FileNotFoundError: No such file: '/home/ssm-user/code/gda/project/tac/data/aclImdb_dataset/dataset_dict.json'. 
Expected to load a `DatasetDict` object, but provided path is not a `DatasetDict`.

In [ ]:
ds["train"].to_parquet("data/aclImdb_train.parquet")
ds["test"].to_parquet("data/aclImdb_test.parquet")

In [51]:
import pandas as pd
from datasets import load_from_disk

In [54]:
train_dataset = Dataset.from_parquet("aclImdb_train.parquet")
test_dataset = Dataset.from_parquet("aclImdb_test.parquet")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [55]:
train_dataset.save_to_disk("train_dataset")
test_dataset.save_to_disk("test_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/25000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/25000 [00:00<?, ? examples/s]

In [56]:
train_dataset = load_from_disk("train_dataset")

# Training job

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [60]:
import boto3
import sagemaker

from models.settings import AWSSettings, DatasetSettings
from sagemaker.huggingface import HuggingFace

aws_settings = AWSSettings()
dataset_settings = DatasetSettings()

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Get the AWS region
region = sagemaker_session.boto_region_name
print(f"SageMaker running in region: {region}")

role = "arn:aws:iam::421646001410:role/service-role/AmazonSageMaker-ExecutionRole-20210811T103532"
print(f"SageMaker role ARN: {aws_settings.EXECUTION_ROLE}")

# Set the bucket name
bucket = "pivanov-tac-bucket"

SageMaker running in region: eu-central-1
SageMaker role ARN: arn:aws:iam::421646001410:role/service-role/AmazonSageMaker-ExecutionRole-20210811T103532


In [61]:
hyperparameters = {
    "epochs": 3,
    "train_batch_size": 32,
    "eval_batch_size": 64,
    "warmup_steps": 500,
    "learning_rate": 5e-5,
    }

In [62]:
huggingface_estimator = HuggingFace(
        entry_point="train.py",
        instance_type="ml.g4dn.2xlarge",
        instance_count=1,
        role=role,
        output_path=f"s3://{bucket}/output/",
        transformers_version="4.6.1",
        pytorch_version="1.7.1",
        py_version="py36",
        hyperparameters = hyperparameters
)

In [63]:
training_parameters = {
    "train": f"s3://{bucket}/datasets/train/",
    "test": f"s3://{bucket}/datasets/test/",
}

In [ ]:
# Run
huggingface_estimator.fit(
    inputs=training_parameters,
    job_name=f"tac-sentiment-analysis-{time.strftime('%Y-%m-%d-%H-%M', time.gmtime())}",
    wait=False
)

[03/21/25 00:46:48] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=631854;file:///home/ssm-user/.cache/pypoetry/virtualenvs/gda-c7vhl4sO-py3.10/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=143000;file:///home/ssm-user/.cache/pypoetry/virtualenvs/gda-c7vhl4sO-py3.10/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     image_uri is not presented, retrieving image_uri based on            ]8;id=82520;file:///home/ssm-user/.cache/pypoetry/virtualenvs/gda-c7vhl4sO-py3.10/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=321108;file:///home/ssm-user/.cache/pypoetry/virtualenvs/gda-c7vhl4sO-py3.10/lib/python3.10/site-packages/sagemaker/image_uris.py#681\681]8;;\
                             instance_type, framework etc.                                                         

                    INFO     Creating training-job with name:                                       ]8;id=997795;file:///home/ssm-user/.cache/pypoetry/virtualenvs/gda-c7vhl4sO-py3.10/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=581891;file:///home/ssm-user/.cache/pypoetry/virtualenvs/gda-c7vhl4sO-py3.10/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             tac-sentiment-analysis-2025-03-20-23-46                                               